In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [2]:
# Esto nos dice que columnas entendio Spark que habia en Mongo
df.printSchema()

# Esto nos muestra las primeras 5 filas para confirmar que hay datos reales
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+------------+--------------------+-----+
|                 _id|categoria|      fecha_captura|       grupo|                item|valor|
+--------------------+---------+-------------------+------------+--------------------+-----+
|69d65793a0ee61da8...|   Travel|2026-04-08 13:26:43|G1_Ecommerce|  It’s Only Himalaya|45.17|
|69d65794a0ee61da8...|   Travel|2026-04-08 13:26:44|G1_Ecommerce|Full Moon over N ...|49.43|
+--------------------+---------+-------------------+------------+--------------------+-----+



In [3]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 2
Registros validos: 2


In [4]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It’s Only Himalaya|45.17|
|Full Moon over N ...|49.43|
+--------------------+-----+



In [5]:
df.filter(df["valor"] > 100).show()

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [6]:
df.groupBy("grupo").count().show()

+------------+-----+
|       grupo|count|
+------------+-----+
|G1_Ecommerce|    2|
+------------+-----+



In [7]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+---------------+-------------+-------------+
|categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+---------------+-------------+-------------+
|   Travel|              2|           47.3|        45.17|        49.43|
+---------+---------------+---------------+-------------+-------------+



In [17]:
from pyspark.sql import functions as F

# 1. Encontramos cuál es el valor máximo de la columna 'valor'
valor_maximo = df.select(F.max("valor")).collect()[0][0]

# 2. Filtramos el DataFrame por ese valor exacto para ver todo el registro
ganador = df.filter(F.col("valor") == valor_maximo)

print(f"VALOR MÁXIMO DETECTADO: {valor_maximo}")
ganador.select("item", "categoria", "valor", "fecha_captura").show(truncate=False)


VALOR MÁXIMO DETECTADO: 49.43
+----------------------------+---------+-----+-------------------+
|item                        |categoria|valor|fecha_captura      |
+----------------------------+---------+-----+-------------------+
|Full Moon over N o a h s Ark|Travel   |49.43|2026-04-08 13:26:44|
+----------------------------+---------+-----+-------------------+



In [19]:
from pyspark.sql import functions as F

# CONSULTA DE SALIDA: Resumen de integridad por Categor a
ticket_salida = df.filter(F.col("grupo") == 'G1_Ecommerce') \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {'G1_Ecommerce'} ---")
ticket_salida.show()

--- TICKET DE SALIDA: G1_Ecommerce ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
|   Travel|           2|        47.3|  2026-04-08 13:26:44|
+---------+------------+------------+---------------------+

